In [16]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

# im2col

In [17]:
def im2col(X, kernel_shape, stride=1, padding=0):
    H, W = X.shape
    kH, kW = kernel_shape

    X_padded = np.pad(X, ((padding, padding), (padding, padding)), mode='constant')

    H_p, W_p = X_padded.shape

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    cols = []

    for i in range(0, out_H*stride, stride):
        for j in range(0, out_W * stride, stride):
            patch = X_padded[i:i+kH, j:j+kW].ravel()
            cols.append(patch)
    
    return np.array(cols), out_H, out_W

def correlate2d_im2col(X, W, stride=1, padding=0):
    X_col, out_H, out_W = im2col(X, W.shape, stride, padding)
    y = X_col @ W.ravel()
    return y.reshape(out_H, out_W)

def im2col_multi(X, kernel_shape, stride=1, padding=0):
    C, H, W = X.shape
    kH, kW = kernel_shape

    X_padded = np.pad(X, ( (0, 0),(padding, padding), (padding, padding) ), mode='constant')

    H_p, W_p = X_padded.shape[1:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    cols = []

    for i in range(0, out_H*stride, stride):
        for j in range(0, out_W * stride, stride):
            patch = X_padded[:, i:i+kH, j:j+kW].ravel()
            cols.append(patch)
    
    return np.array(cols), out_H, out_W

def col2im_multi(cols, output_shape, kernel_shape, stride=1, padding=0):
    C, H, W = output_shape
    kH, kW = kernel_shape
    H_p, W_p = H+2*padding, W+2*padding
    X_padded = np.zeros((C,H_p,W_p))

    out_H = (H_p - kH)//stride + 1
    out_W = (W_p - kW)//stride + 1

    idx = 0
    for i in range(0, out_H*stride, stride):
        for j in range(0, out_W*stride, stride):
            patch = cols[idx].reshape(C, kH, kW)
            X_padded[:, i:i+kH, j:j+kW] += patch
            idx += 1

    if padding>0:
        X_padded = X_padded[:, padding:-padding, padding:-padding]

    return X_padded

def conv2d_im2col_multi(X, W, stride=1, padding=0):
    
    C_out, C_in, kH, kW = W.shape
    X_col, out_H, out_W = im2col_multi(X, (kH, kW), stride, padding)

    W_col = W.reshape(C_out, -1)
    Y_col = X_col @ W_col.T

    Y = Y_col.T.reshape(C_out, out_H, out_W)
    return Y

def conv_transpose2d_img2col_multi(Y, W, stride=1, padding=0, output_shape=None):
    C_out, C_in, kH, kW = W.shape
    Y_col = Y.reshape(C_out, -1)
    W_col = W.reshape(C_out, -1)
    X_col = W_col.T @ Y_col

    if output_shape is None:
        H_out = (Y.shape[1]-1) * stride - 2*padding + kH
        W_out = (Y.shape[2]-1) * stride - 2*padding + kW
        output_shape = (C_in, H_out, W_out)

    X = col2im_multi(X_col.T, output_shape=output_shape, kernel_shape=(kH, kW), stride=stride, padding=padding)

    return X

In [18]:
np.random.seed(1)

padding=1
stride=2

X = np.arange(16).reshape(4, 4)
#X = np.pad(X, ((padding, padding), (padding, padding)), mode='constant')
kernel = np.random.randn(2, 2)

#print(X)

In [19]:
new, out_H, out_W = im2col(X, kernel_shape=(2, 2), stride=1, padding=0)
#print(new)

In [20]:
result = (new @ kernel.reshape(-1)).reshape(out_H, out_W)
#print(result)

In [21]:
result = signal.correlate2d(X, kernel, mode='valid')
#print(result)

In [22]:
result = correlate2d_im2col(X, kernel, stride=1, padding=0)
#print(result)

# Multi

In [23]:
C_out, C_in = 6, 3
input_size=15
kernel_size=3

X = np.random.randn(C_in, input_size, input_size)
kernel = np.random.randn(C_out, C_in, kernel_size, kernel_size)

print(X.shape)

(3, 15, 15)


In [24]:
new, out_H, out_W = im2col_multi(X, (kernel_size, kernel_size))
print(new.shape)

(169, 27)


In [25]:
results = []
for i in range(C_out):
    temp = (new @ kernel[i].ravel()).reshape(out_H, out_W)
    results.append(temp)

results = np.array(results)
print(results[0][0])

[ 0.93656834  5.16907291  2.71131652  8.9836369   0.95733795 -2.14783409
 -4.5897962  -0.85836645  0.56491241 -0.49184751 -0.68513367 -4.4240134
 -1.96562923]


In [26]:
result = np.zeros((C_out, out_H, out_W))

for i in range(C_in):
    for j in range(C_out):
        result[j] += signal.correlate2d(X[i], kernel[j, i], mode='valid')

print(result[0][0])

[ 0.93656834  5.16907291  2.71131652  8.9836369   0.95733795 -2.14783409
 -4.5897962  -0.85836645  0.56491241 -0.49184751 -0.68513367 -4.4240134
 -1.96562923]


In [27]:
stride=1
padding=0

result = conv2d_im2col_multi(X, kernel, stride=stride, padding=padding)

print(result[0][0])

[ 0.93656834  5.16907291  2.71131652  8.9836369   0.95733795 -2.14783409
 -4.5897962  -0.85836645  0.56491241 -0.49184751 -0.68513367 -4.4240134
 -1.96562923]


In [28]:
print(f'Input: {X.shape}')
print(f'Kernel: {kernel.shape}')
print(f'Result: {result.shape}')

Input: (3, 15, 15)
Kernel: (6, 3, 3, 3)
Result: (6, 13, 13)


In [29]:
C_out, C_in, kH, kW = kernel.shape
Y_col = result.reshape(C_out, -1)
print(f'Y_col: {Y_col.shape}')
W_col = kernel.reshape(C_out, -1)
print(f'W_col: {W_col.shape}')
X_col = W_col.T @ Y_col
print(f'X_col: {X_col.shape}')

X = col2im_multi(X_col.T, output_shape=(3, 4, 4), kernel_shape=(kH, kW), stride=1, padding=0)

print(f'X: {X.shape}')

Y_col: (6, 169)
W_col: (6, 27)
X_col: (27, 169)
X: (3, 4, 4)


In [30]:
X = conv_transpose2d_img2col_multi(result, kernel, stride=stride, padding=padding, output_shape=None)
print(X.shape)

(3, 15, 15)
